## 1. Tooth Detection and Segmentation
Process dental cases with the Tooth Segmentation + Recognition model.
This step outputs bounding-box images, mask overlays, and per-case JSON files.

Next step output used by Step 2:
- segmentation+recognition-dataset

In [2]:
# Step 1: Tooth Detection and Segmentation on raw_data (all cases)
import os
import json
import random
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

# IMPORTANT for Windows/Jupyter: use non-interactive backend to avoid Tkinter issues
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ultralytics import YOLO
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    union = float(areaA + areaB - inter)
    return inter / union if union > 0 else 0.0


def get_device():
    import torch

    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using device: cuda | GPU: {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
        print("Using device: mps")
    else:
        device = "cpu"
        print("Using device: cpu")
    return device


def get_detectron2_device(device: str) -> str:
    # Detectron2 reliably supports cuda/cpu. If main device is mps, use cpu for Detectron2.
    if device == "cuda":
        return "cuda"
    if device == "mps":
        print("Detectron2 uses cpu because mps support is limited.")
        return "cpu"
    return "cpu"


def initialize_models(raw_data_dir: Path):
    device = get_device()
    detectron_device = get_detectron2_device(device)

    # Try common locations for model weights (supports both 'weights' and 'weights(model)')
    yolo_candidates = [
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_pano_20250319.pt",
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_pano_20250319.pt",
        Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_pano_20250319.pt",
        Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_pano_20250319.pt",
    ]
    detectron_candidates = [
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_crop_20250424.pth",
        raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_crop_20250424.pth",
        Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_crop_20250424.pth",
        Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_crop_20250424.pth",
    ]

    yolo_path = next((p for p in yolo_candidates if p.exists()), None)
    detectron_path = next((p for p in detectron_candidates if p.exists()), None)

    if yolo_path is None or detectron_path is None:
        checked = [str(p) for p in (yolo_candidates + detectron_candidates)]
        raise FileNotFoundError(
            "Model weights not found. Checked paths:\n" + "\n".join(checked)
        )

    print(f"YOLO weights: {yolo_path}")
    print(f"Detectron2 weights: {detectron_path}")

    yolo_model = YOLO(str(yolo_path))
    yolo_model.to(device)

    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.WEIGHTS = str(detectron_path)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.DEVICE = detectron_device
    detectron_predictor = DefaultPredictor(cfg)

    print("Models initialized successfully")
    return yolo_model, detectron_predictor


def process_panoramic_image(pano_img_rgb, yolo_model):
    results = yolo_model(pano_img_rgb, verbose=False)
    h, w = pano_img_rgb.shape[:2]
    all_masks = []

    for result in results:
        if result.masks is None or not hasattr(result.masks, "data"):
            continue

        temp = []
        num_masks = len(result.masks.xy) if hasattr(result.masks, "xy") else 0

        for i in range(num_masks):
            mask_matrix = result.masks.data[i].cpu().numpy()
            resized_mask = cv2.resize(
                mask_matrix.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST
            ).astype(bool)

            bbox = result.boxes.xyxy[i].cpu().numpy().tolist()
            cls_id = int(result.boxes.cls[i].item())

            temp.append({
                "id": i,
                "matrix_resized": resized_mask,
                "confidence": float(result.boxes.conf[i].item()),
                "class_id": cls_id,
                "class_name": str(result.names[cls_id]),
                "bbox": bbox,
            })

        # Remove duplicates by IoU
        kept = []
        for m in temp:
            duplicated = False
            for k in kept:
                if compute_iou(m["bbox"], k["bbox"]) > 0.5:
                    if m["confidence"] > k["confidence"]:
                        kept.remove(k)
                        kept.append(m)
                    duplicated = True
                    break
            if not duplicated:
                kept.append(m)

        all_masks.extend(kept)

    return all_masks


def crop_and_segment_teeth(pano_img_rgb, mask_data_list, detectron_predictor, pad=20):
    outputs = []
    for data in mask_data_list:
        mask = data["matrix_resized"].astype(np.uint8) * 255
        x, y, w, h = cv2.boundingRect(mask)
        x1 = max(x - pad, 0)
        y1 = max(y - pad, 0)
        x2 = min(x + w + pad, pano_img_rgb.shape[1])
        y2 = min(y + h + pad, pano_img_rgb.shape[0])

        cropped = pano_img_rgb[y1:y2, x1:x2]
        pred = detectron_predictor(cropped)
        instances = pred["instances"].to("cpu")

        segments = []
        all_pixels = []
        if len(instances) > 0:
            masks = instances.pred_masks.numpy()
            scores = instances.scores.numpy()
            for idx, (seg_mask, score) in enumerate(zip(masks, scores)):
                rows, cols = np.where(seg_mask)
                pixel_coords = [(int(x1 + c), int(y1 + r)) for r, c in zip(rows, cols)]
                all_pixels.extend(pixel_coords)
                segments.append({
                    "mask_id": idx,
                    "score": float(score),
                    "num_pixels": len(pixel_coords),
                    "pixel_coordinates": pixel_coords,
                })

        outputs.append({
            "tooth_id": data["class_name"],
            "crop_coords": [int(x1), int(y1), int(x2), int(y2)],
            "num_segments": len(segments),
            "segments": segments,
            "pixel_coordinates": all_pixels,
            "total_pixels": len(all_pixels),
            "detectron_output": pred,
        })

    return outputs


def save_bbox_visualization(pano_img_rgb, list_of_masks, output_path):
    vis = pano_img_rgb.copy()
    for m in list_of_masks:
        x1, y1, x2, y2 = map(int, m["bbox"])
        label = str(m["class_name"])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    fig = plt.figure(figsize=(12, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Bounding Boxes with Tooth ID")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def save_mask_overlay(pano_img_rgb, list_seg_tooth, output_path, alpha=0.5):
    vis = pano_img_rgb.copy()

    for result in list_seg_tooth:
        x1, y1, x2, y2 = result["crop_coords"]
        pred = result["detectron_output"]
        instances = pred["instances"].to("cpu")

        if len(instances) == 0:
            continue

        masks = instances.pred_masks.numpy()
        for mask in masks:
            h, w = mask.shape
            color = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.uint8)

            color_mask = np.zeros((h, w, 3), dtype=np.uint8)
            color_mask[mask] = color

            crop = vis[y1:y2, x1:x2]
            blended = np.where(
                mask[:, :, None],
                cv2.addWeighted(crop, 1 - alpha, color_mask, alpha, 0),
                crop,
            )
            vis[y1:y2, x1:x2] = blended

    fig = plt.figure(figsize=(16, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.title("Mask Overlay")
    plt.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.close(fig)


def save_results_json(case_num, list_of_masks, list_seg_tooth, output_dir):
    result = {
        "case_number": str(case_num),
        "num_teeth_detected": len(list_of_masks),
        "teeth_data": [],
    }

    for mask_data, seg_data in zip(list_of_masks, list_seg_tooth):
        result["teeth_data"].append({
            "tooth_id": mask_data["class_name"],
            "confidence": float(mask_data["confidence"]),
            "bbox": [float(v) for v in mask_data["bbox"]],
            "crop_coords": seg_data["crop_coords"],
            "num_segments": seg_data["num_segments"],
            "total_pixels": seg_data["total_pixels"],
            "pixel_coordinates": seg_data["pixel_coordinates"],
            "segments_detail": seg_data["segments"],
        })

    with open(output_dir / f"case_{case_num}_results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def process_case(case_dir, yolo_model, detectron_predictor, output_root):
    case_num = case_dir.name.replace("case ", "")
    png_path = case_dir / f"case_{case_num}.png"
    if not png_path.exists():
        png_files = sorted(case_dir.glob("*.png"))
        if not png_files:
            return False, "No PNG found"
        png_path = png_files[0]

    img_bgr = cv2.imread(str(png_path))
    if img_bgr is None:
        return False, "Image read failed"

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    list_of_masks = process_panoramic_image(img_rgb, yolo_model)
    if len(list_of_masks) == 0:
        return False, "No teeth detected"

    list_seg_tooth = crop_and_segment_teeth(img_rgb, list_of_masks, detectron_predictor)

    case_out = output_root / f"case {case_num}"
    case_out.mkdir(parents=True, exist_ok=True)

    save_bbox_visualization(img_rgb, list_of_masks, case_out / f"case_{case_num}_bounding_boxes.png")
    save_mask_overlay(img_rgb, list_seg_tooth, case_out / f"case_{case_num}_mask_overlay.png")
    save_results_json(case_num, list_of_masks, list_seg_tooth, case_out)

    return True, f"OK ({len(list_of_masks)} teeth)"


def run_inference_on_raw_dataset():
    project_root = Path("E:/MU/3rd-year/Project_senior")
    raw_data_dir = project_root / "raw_data"
    cases_root = raw_data_dir / "500 cases with annotation(xml)"
    # cases_root = Path("/Users/sukollapat/Desktop/pond_work/mahidol_work/dst_mahidol/senior_pj/data/500 cases with annotation-raw")

    if not cases_root.exists():
        raise FileNotFoundError(f"Cases folder not found: {cases_root}")

    output_root = project_root / "segmentation+recognition-dataset"
    output_root.mkdir(parents=True, exist_ok=True)

    yolo_model, detectron_predictor = initialize_models(raw_data_dir)

    case_dirs = sorted(
        [d for d in cases_root.iterdir() if d.is_dir() and d.name.startswith("case ")],
        key=lambda p: int(p.name.replace("case ", "")),
    )

    ok_count = 0
    fail_count = 0
    print(f"Processing cases: {len(case_dirs)}")

    for case_dir in tqdm(case_dirs, desc="raw_data cases", unit="case"):
        try:
            ok, msg = process_case(case_dir, yolo_model, detectron_predictor, output_root)
            if ok:
                ok_count += 1
            else:
                fail_count += 1
        except Exception as e:
            fail_count += 1
            print(f"Error in {case_dir.name}: {e}")

    summary = {
        "source": "raw_data/500 cases with annotation(xml)",
        "processed": ok_count,
        "failed": fail_count,
        "total": len(case_dirs),
    }

    with open(output_root / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nDone")
    print(f"Output folder: {output_root.resolve()}")
    print(json.dumps(summary, indent=2))


run_inference_on_raw_dataset()

/Users/jaopichitchai/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'ultralytics'

## 2. Map tooth positions to caries regions (JSON output)
Combined script that processes dental cases to:
1. Map tooth positions to caries regions (JSON output)
2. Generate visual alignment debug images (PNG output)

Data Sources:
- JSON: segmentation+recognition-dataset/case X/case_X_results.json
  Contains pixel_coordinates (list of [x, y]) for each tooth
- ROI Image: raw_data/500-roi/case_X.png
  Binary mask where non-zero = caries, 0 = normal

Output (single root folder):
- caries_mapping_output/case X/case_X_caries_mapping.json
- caries_mapping_output/case X/case_X_alignment_detailed.png
- caries_mapping_output/caries_mapping_results.csv (summary)

In [4]:
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


# =============================================================================
# ROI & JSON Loading Functions
# =============================================================================

def load_roi_image(roi_path: Path):
    """Load ROI mask and normalize to 2D grayscale array."""
    roi_img = cv2.imread(str(roi_path), cv2.IMREAD_UNCHANGED)
    if roi_img is None:
        raise FileNotFoundError(f"Could not load ROI image: {roi_path}")

    # Some PNGs are loaded as (H, W, 1); convert to strict 2D for downstream ops.
    if roi_img.ndim == 3:
        if roi_img.shape[2] == 1:
            roi_img = roi_img[:, :, 0]
        else:
            roi_img = cv2.cvtColor(roi_img, cv2.COLOR_BGR2GRAY)

    return roi_img


def load_tooth_json(json_path: Path):
    """Load tooth segmentation JSON with pixel coordinates."""
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# Caries Mapping Functions
# =============================================================================

def calculate_caries_overlap(roi_img, pixel_coordinates):
    """Calculate overlap between tooth pixels and caries region in ROI."""
    if not pixel_coordinates:
        return 0, 0, 0.0, []

    height, width = roi_img.shape[:2]
    caries_count = 0
    caries_coords = []
    valid_pixels = 0

    for coord in pixel_coordinates:
        x, y = coord[0], coord[1]

        if 0 <= x < width and 0 <= y < height:
            valid_pixels += 1
            if roi_img[y, x] > 0:
                caries_count += 1
                caries_coords.append([x, y])

    percentage = (caries_count / valid_pixels * 100) if valid_pixels > 0 else 0.0
    return caries_count, valid_pixels, percentage, caries_coords


def generate_caries_mapping(roi_img, tooth_data, case_num):
    """Generate per-tooth caries mapping results for one case."""
    results = []

    for tooth in tooth_data.get("teeth_data", []):
        tooth_id = tooth.get("tooth_id", "unknown")
        pixel_coords = tooth.get("pixel_coordinates", [])
        confidence = tooth.get("confidence", 0.0)

        caries_count, total_pixels, percentage, caries_coords = calculate_caries_overlap(
            roi_img, pixel_coords
        )

        results.append(
            {
                "case_number": int(case_num),
                "tooth_id": tooth_id,
                "confidence": float(confidence),
                "total_pixels": int(total_pixels),
                "caries_pixels": int(caries_count),
                "caries_percentage": round(float(percentage), 4),
                "has_caries": bool(caries_count > 0),
                "caries_coordinates": caries_coords,
            }
        )

    return results


# =============================================================================
# Visual Alignment Functions
# =============================================================================

def create_alignment_visualization(roi_img, tooth_data, alpha=0.5):
    """Create blended visualization: ROI background + tooth pixels overlay."""
    height, width = roi_img.shape[:2]

    roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)
    roi_bgr = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
    tooth_overlay = np.zeros((height, width, 3), dtype=np.uint8)

    colors = [
        (0, 255, 0),
        (0, 255, 255),
        (255, 255, 0),
        (0, 165, 255),
        (255, 0, 255),
        (128, 255, 0),
        (255, 128, 0),
        (0, 128, 255),
    ]

    tooth_count = 0
    total_pixels_drawn = 0
    out_of_bounds = 0

    for tooth in tooth_data.get("teeth_data", []):
        pixel_coords = tooth.get("pixel_coordinates", [])
        color = colors[tooth_count % len(colors)]
        tooth_count += 1

        for coord in pixel_coords:
            x, y = coord[0], coord[1]
            if 0 <= x < width and 0 <= y < height:
                tooth_overlay[y, x] = color
                total_pixels_drawn += 1
            else:
                out_of_bounds += 1

    tooth_mask = np.any(tooth_overlay > 0, axis=2)
    result = roi_bgr.copy()
    blended = cv2.addWeighted(roi_bgr, 1 - alpha, tooth_overlay, alpha, 0)
    result[tooth_mask] = blended[tooth_mask]
    strong_overlay = cv2.addWeighted(result, 0.7, tooth_overlay, 0.3, 0)
    result[tooth_mask] = strong_overlay[tooth_mask]

    return result, {
        "teeth_count": tooth_count,
        "pixels_drawn": total_pixels_drawn,
        "out_of_bounds": out_of_bounds,
        "roi_shape": (height, width),
    }


def create_detailed_alignment_image(roi_img, tooth_data):
    """Create 3-panel alignment image for QA/debug."""
    height, width = roi_img.shape[:2]
    roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)

    roi_colored = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
    caries_mask = roi_img > 0
    roi_colored[caries_mask] = [0, 0, 255]

    tooth_only = np.zeros((height, width, 3), dtype=np.uint8)
    for tooth in tooth_data.get("teeth_data", []):
        for coord in tooth.get("pixel_coordinates", []):
            x, y = coord[0], coord[1]
            if 0 <= x < width and 0 <= y < height:
                tooth_only[y, x] = [0, 255, 0]

    blended, stats = create_alignment_visualization(roi_img, tooth_data, alpha=0.6)

    tooth_mask = np.any(tooth_only > 0, axis=2)
    overlap_pixels = int(np.sum(tooth_mask & caries_mask))
    caries_pixels = int(np.sum(caries_mask))
    tooth_pixels = int(np.sum(tooth_mask))

    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(roi_colored, "ROI (Red=Caries)", (20, 60), font, 1.5, (255, 255, 255), 3)
    cv2.putText(tooth_only, "Teeth (Green)", (20, 60), font, 1.5, (255, 255, 255), 3)
    cv2.putText(blended, "ALIGNMENT CHECK", (20, 60), font, 1.5, (0, 255, 255), 3)

    stats_text = [
        f"Teeth: {stats['teeth_count']}",
        f"Tooth pixels: {tooth_pixels:,}",
        f"Caries pixels: {caries_pixels:,}",
        f"Overlap: {overlap_pixels:,}",
    ]
    y_offset = 120
    for text in stats_text:
        cv2.putText(blended, text, (20, y_offset), font, 1.0, (255, 255, 0), 2)
        y_offset += 40

    scale = 0.5
    roi_small = cv2.resize(roi_colored, None, fx=scale, fy=scale)
    tooth_small = cv2.resize(tooth_only, None, fx=scale, fy=scale)
    blended_small = cv2.resize(blended, None, fx=scale, fy=scale)
    combined = np.hstack([roi_small, tooth_small, blended_small])

    overlap_stats = {
        "overlap_pixels": overlap_pixels,
        "caries_pixels": caries_pixels,
        "tooth_pixels": tooth_pixels,
    }
    return combined, stats, overlap_stats


# =============================================================================
# Pipeline Functions for Current Project Structure
# =============================================================================

def parse_case_num_from_json(json_path: Path):
    """Extract numeric case id from filename: case_123_results.json -> 123."""
    stem = json_path.stem
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected JSON filename format: {json_path.name}")
    return int(parts[1])


def collect_case_records(segmentation_root: Path, roi_root: Path):
    """Collect case records from Step 1 output and matching ROI paths."""
    records = []

    case_dirs = sorted(
        [p for p in segmentation_root.iterdir() if p.is_dir() and p.name.startswith("case ")],
        key=lambda p: int(p.name.replace("case ", "")),
    )

    for case_dir in case_dirs:
        json_candidates = sorted(case_dir.glob("case_*_results.json"))
        if not json_candidates:
            continue

        json_path = json_candidates[0]
        case_num = parse_case_num_from_json(json_path)
        roi_path = roi_root / f"case_{case_num}.png"

        records.append(
            {
                "case_number": case_num,
                "json_path": json_path,
                "roi_path": roi_path,
            }
        )

    return records


def process_single_case(record, output_root: Path):
    """Process one case and save both JSON + alignment image in one case folder."""
    case_num = record["case_number"]
    json_path = record["json_path"]
    roi_path = record["roi_path"]

    if not json_path.exists():
        return False, "JSON not found", None
    if not roi_path.exists():
        return False, f"ROI not found: {roi_path.name}", None

    roi_img = load_roi_image(roi_path)
    tooth_data = load_tooth_json(json_path)
    caries_results = generate_caries_mapping(roi_img, tooth_data, case_num)

    case_output_dir = output_root / f"case {case_num}"
    case_output_dir.mkdir(parents=True, exist_ok=True)

    caries_json_path = case_output_dir / f"case_{case_num}_caries_mapping.json"
    with open(caries_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "case_number": int(case_num),
                "teeth_caries_data": caries_results,
            },
            f,
            indent=2,
        )

    combined_img, stats, overlap_stats = create_detailed_alignment_image(roi_img, tooth_data)
    alignment_img_path = case_output_dir / f"case_{case_num}_alignment_detailed.png"
    cv2.imwrite(str(alignment_img_path), combined_img)

    teeth_with_caries = sum(1 for r in caries_results if r["has_caries"])
    msg = f"{len(caries_results)} teeth, {teeth_with_caries} with caries"

    case_stats = {
        "stats": stats,
        "overlap_stats": overlap_stats,
        "caries_results": caries_results,
    }
    return True, msg, case_stats


def run_caries_mapping_pipeline(
    segmentation_root: Path,
    roi_root: Path,
    output_root: Path,
):
    """Run mapping pipeline and keep all outputs under one root folder."""
    output_root.mkdir(parents=True, exist_ok=True)

    records = collect_case_records(segmentation_root, roi_root)
    if not records:
        raise FileNotFoundError(
            "No case JSON files found. Check segmentation_root path and Step 1 outputs."
        )

    summary_stats = {
        "total_cases": len(records),
        "processed_cases": 0,
        "failed_cases": 0,
        "total_teeth": 0,
        "teeth_with_caries": 0,
    }
    all_caries_results = []

    print("=" * 70)
    print("Dental Caries Mapping Pipeline")
    print("=" * 70)
    print(f"Segmentation JSON source: {segmentation_root}")
    print(f"ROI source: {roi_root}")
    print(f"Output root: {output_root}")
    print("=" * 70)

    pbar = tqdm(records, desc="Processing cases", unit="case")
    for record in pbar:
        case_num = record["case_number"]
        try:
            success, msg, payload = process_single_case(record, output_root=output_root)

            if success and payload:
                case_results = payload["caries_results"]
                summary_stats["processed_cases"] += 1
                summary_stats["total_teeth"] += len(case_results)
                summary_stats["teeth_with_caries"] += sum(
                    1 for r in case_results if r["has_caries"]
                )

                for r in case_results:
                    csv_result = {k: v for k, v in r.items() if k != "caries_coordinates"}
                    all_caries_results.append(csv_result)

                pbar.set_postfix_str(f"case {case_num}: {msg}")
            else:
                summary_stats["failed_cases"] += 1
                pbar.set_postfix_str(f"case {case_num}: FAIL - {msg}")

        except Exception as e:
            summary_stats["failed_cases"] += 1
            pbar.set_postfix_str(f"case {case_num}: ERROR - {str(e)[:40]}")

    if all_caries_results:
        df = pd.DataFrame(all_caries_results)
        csv_path = output_root / "caries_mapping_results.csv"
        df.to_csv(csv_path, index=False)
        print(f"\nResults CSV saved to: {csv_path}")

        summary_df = (
            df.groupby(["case_number"]).agg(
                {
                    "tooth_id": "count",
                    "caries_pixels": "sum",
                    "has_caries": "sum",
                    "caries_percentage": "mean",
                }
            ).rename(
                columns={
                    "tooth_id": "total_teeth",
                    "has_caries": "teeth_with_caries",
                    "caries_percentage": "avg_caries_percentage",
                }
            )
        )
        summary_csv_path = output_root / "caries_summary_by_case.csv"
        summary_df.to_csv(summary_csv_path)
        print(f"Summary CSV saved to: {summary_csv_path}")

    print("\n" + "=" * 70)
    print("Processing Complete")
    print("=" * 70)
    print(f"Total cases found: {summary_stats['total_cases']}")
    print(f"Processed: {summary_stats['processed_cases']}")
    print(f"Failed: {summary_stats['failed_cases']}")
    print(f"Total teeth analyzed: {summary_stats['total_teeth']}")
    print(f"Teeth with caries: {summary_stats['teeth_with_caries']}")
    if summary_stats["total_teeth"] > 0:
        pct = summary_stats["teeth_with_caries"] / summary_stats["total_teeth"] * 100
        print(f"Caries prevalence: {pct:.2f}%")
    print("=" * 70)

    return all_caries_results, summary_stats


# =============================================================================
# Notebook Runner (Current Project Environment)
# =============================================================================

project_root = Path.cwd()

# Step 1 output folder (refactored pipeline)
segmentation_root = project_root / "segmentation+recognition-dataset"
if not segmentation_root.exists():
    raise FileNotFoundError(
        f"Segmentation output folder not found: {segmentation_root}. Run Step 1 first."
    )

# ROI folder from raw_data
roi_root = project_root / "raw_data" / "500-roi"
if not roi_root.exists():
    raise FileNotFoundError(
        f"ROI folder not found: {roi_root}."
    )

# Single output root as requested
output_root = project_root / "caries_mapping_output"

all_caries_results, summary_stats = run_caries_mapping_pipeline(
    segmentation_root=segmentation_root,
    roi_root=roi_root,
    output_root=output_root,
)

/home/phaihome/miniconda3/envs/dental/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: Segmentation output folder not found: /mnt/e/MU/3rd-year/Senior/dental-caries-detection/weekphase2/segmentation+recognition-dataset. Run Step 1 first.

## 3. PCA-based tooth alignment + 3-surface classification (Visualization Output)

This module extends the pipeline by applying PCA-based alignment
to each tooth and classifying caries into 3 surfaces:
- Occlusal
- Mesial
- Distal

It also generates visualization outputs for debugging and interpretation.

Output Structure:
- PCA_Output/case X/
    <!-- - case_X_pca_alignment.png -->
    - case_X_pca_results.json

## 4. Visualization of PCA-Aligned Teeth and Caries Surfaces

This module provides visualization functions for each tooth after PCA alignment.
It supports:

Showing aligned tooth polygon
Displaying Occlusal / Mesial / Distal zones
Plotting caries pixels and centroid
Drawing bounding box for reference
Generating grid layout for all teeth in a case

Output can be saved as .png for reporting or debugging.


In [3]:
# =============================================================================
# PCA & Surface Classification + Visualization
# =============================================================================

from ast import Tuple
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

# =============================================================================
# CONFIG
# =============================================================================
SEG_DIR = "segmentation+recognition-dataset"
CARIES_DIR = "caries_mapping_output"
OUT_DIR = "PCA_Output"
os.makedirs(OUT_DIR, exist_ok=True)

OCCLUSAL_RATIO = 0.2
SIDE_RATIO = 0.2

# Surface color mapping
SURFACE_COLORS = {
    "Occlusal": '#E74C3C',  # Red
    "Mesial": '#3498DB',    # Blue
    "Distal": '#3498DB',     # Blue
    "Other": '#2ECC71',      # Green
    -1: '#95A5A6'            # Gray
}

SURFACE_LABELS = {
    "Occlusal": "Occlusal",
    "Mesial": "Mesial",
    "Distal": "Distal",
    "Other": "Other"
}

# =============================================================================
# HELPERS
# =============================================================================
def is_upper_jaw(tid):
    return int(str(tid)[0]) in [1, 2]

def load_seg(case_id):
    path = os.path.join(SEG_DIR, f"case {case_id}", f"case_{case_id}_results.json")
    if not os.path.exists(path):
        print(f"[ERROR] SEG missing: {path}")
        return None
    return json.load(open(path))

def load_caries(case_id):
    path = os.path.join(CARIES_DIR, f"case {case_id}", f"case_{case_id}_caries_mapping.json")
    if not os.path.exists(path):
        print(f"[ERROR] CARIES missing: {path}")
        return None
    return json.load(open(path))

def build_seg_map(seg_data):
    mapping = {}
    for t in seg_data.get("teeth_data", []):
        tid = str(t["tooth_id"])
        mapping[tid] = t.get("pixel_coordinates", [])
    return mapping

def get_caries_list(caries_data):
    return caries_data.get("teeth_caries_data", [])

def compute_centroid(points):
    pts = np.array(points)
    return np.mean(pts[:, 0]), np.mean(pts[:, 1])

def get_bbox(points):
    pts = np.array(points)
    x_min, y_min = np.min(pts, axis=0)
    x_max, y_max = np.max(pts, axis=0)
    return x_min, y_min, x_max - x_min, y_max - y_min

# =============================================================================
# PCA & Rotation
# =============================================================================
# def perform_pca(points, tid):
#     pts = np.array(points, dtype=np.float64)
#     mean = np.mean(pts, axis=0)
#     centered = pts - mean
#     cov = np.cov(centered, rowvar=False)
#     eigvals, eigvecs = np.linalg.eigh(cov)
#     idx = np.argsort(eigvals)[::-1]
#     major = eigvecs[:, idx[0]]
#     major = major / np.linalg.norm(major)

#     # Fix direction
#     if is_upper_jaw(tid):
#         if major[1] < 0: major = -major
#     else:
#         if major[1] > 0: major = -major

#     angle = np.arctan2(major[1], major[0])
#     rotation_angle = (np.pi / 2) - angle
#     return mean, rotation_angle

def perform_pca(points, tooth_id):
    """
    Apply PCA to align the tooth vertically using heuristics for square shapes 
    and FDI notation to orient the major axis towards the occlusal surface.
    """
    # 1. หาจุดกึ่งกลางและ Centering
    mean = np.mean(points, axis=0)
    centered_data = points - mean
    
    # 2. คำนวณ Covariance และ Eigenvectors
    cov_matrix = np.cov(centered_data, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # ดึงเวกเตอร์ทั้ง 2 ตัวออกมา
    vec_0 = eigenvectors[:, 0]
    vec_1 = eigenvectors[:, 1]
    
    # ตรวจสอบว่าฟันเป็น "ทรงจตุรัส" หรือไม่?
    ratio = eigenvalues[1] / (eigenvalues[0] + 1e-6) 
    is_square_like = ratio < 2.0  
    
    if is_square_like:
        # กรณีฟันป้อม/จตุรัส: เลือกเวกเตอร์ที่มีค่า y (แนวตั้ง) มากกว่า เป็นแกนหลักเสมอ
        if abs(vec_0[1]) > abs(vec_1[1]):
            major_axis = vec_0
            minor_axis = vec_1
        else:
            major_axis = vec_1
            minor_axis = vec_0
    else:
        # กรณีฟันยาวปกติ: เลือกตาม Eigenvalue มากที่สุดเหมือนเดิม
        sort_indices = np.argsort(eigenvalues)[::-1]
        major_axis = eigenvectors[:, sort_indices[0]]
        minor_axis = eigenvectors[:, sort_indices[1]]

    # --- [ส่วนที่เพิ่มใหม่] ล็อคทิศทาง Vector ให้พุ่งเข้าหา Occlusal (กึ่งกลางปาก) ---
    
    # วิเคราะห์ว่าเป็นฟันบนหรือฟันล่างจาก FDI Notation (ตัวเลขแรก)
    is_upper_jaw = True
    if tooth_id and len(tooth_id) >= 1:
        try:
            quadrant = int(tooth_id[0])
            if quadrant in [3, 4]:
                is_upper_jaw = False
        except ValueError:
            pass # ถ้า Parse ไม่ได้ ให้ถือเป็นฟันบนไปก่อน (หรือจะ raise error ก็ได้)

    # ปรับทิศทางของ Vector ให้ถูกต้องตามพิกัดภาพ 2D
    if is_upper_jaw:
        # ฟันบน Occlusal อยู่ด้านล่างภาพ -> เวกเตอร์ต้องชี้ลง (ค่า Y ต้องเป็นบวก)
        if major_axis[1] < 0:  
            major_axis = -major_axis
            minor_axis = -minor_axis # สลับ minor ด้วยเพื่อรักษาพิกัดมือขวา (Right-hand rule)
    else:
        # ฟันล่าง Occlusal อยู่ด้านบนภาพ -> เวกเตอร์ต้องชี้ขึ้น (ค่า Y ต้องเป็นลบ)
        if major_axis[1] > 0:  
            major_axis = -major_axis
            minor_axis = -minor_axis

    # จัดเรียง Eigenvectors ใหม่ให้ถูกต้อง
    # eigenvectors_sorted = np.column_stack((minor_axis, major_axis))

    # 3. คำนวณมุม
    rotation_angle = np.arctan2(major_axis[1], major_axis[0])
    
    return mean, rotation_angle


# =============================================================================


def rotate(points, center, angle):
    pts = np.array(points)
    c, s = np.cos(angle), np.sin(angle)
    pts_centered = pts - center
    return np.dot(pts_centered, np.array([[c, -s], [s, c]])) + center

# =============================================================================
# Surface classification
# =============================================================================
def classify_surface_full(tid, tooth_pts, caries_pts):
    center, angle = perform_pca(tooth_pts, tid)
    tooth_rot = rotate(tooth_pts, center, angle)
    caries_rot = rotate(caries_pts, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    cx, cy = compute_centroid(caries_rot)

    rel_x = (cx - x) / w
    rel_y = (cy - y) / h

    if is_upper_jaw(tid):
        if rel_y >= (1 - OCCLUSAL_RATIO):
            surface = "Occlusal"
        elif rel_x <= SIDE_RATIO:
            surface = "Mesial"
        elif rel_x >= (1 - SIDE_RATIO):
            surface = "Distal"
        else:
            surface = "Other"
    else:
        if rel_y <= OCCLUSAL_RATIO:
            surface = "Occlusal"
        elif rel_x <= SIDE_RATIO:
            surface = "Mesial"
        elif rel_x >= (1 - SIDE_RATIO):
            surface = "Distal"
        else:
            surface = "Other"

    return surface, float(np.degrees(angle))

def compute_caries_stats(caries_pts, tooth_pts):
    caries_pixels = len(caries_pts)
    tooth_pixels = len(tooth_pts)
    if tooth_pixels == 0: return 0, 0
    percentage = (caries_pixels / tooth_pixels) * 100
    return caries_pixels, percentage

# =============================================================================
# Visualization
# =============================================================================
def visualize_tooth_with_zones(tooth_pts, caries_pts, tooth_id, classification,
                               occ_ratio=OCCLUSAL_RATIO, side_ratio=SIDE_RATIO, save_path=None):
    tooth_pts = np.array(tooth_pts, dtype=np.float64)
    caries_pts = np.array(caries_pts, dtype=np.float64)
    
    x, y = np.min(tooth_pts, axis=0)
    w, h = np.ptp(tooth_pts, axis=0)
    
    fig, ax = plt.subplots(figsize=(6,6))
    
    # --- Draw zones (background, semi-transparent) ---
    if is_upper_jaw(tooth_id):
        occl_y_start = y + h*(1-occ_ratio)
        occl_y_end = y + h
    else:
        occl_y_start = y
        occl_y_end = y + h*occ_ratio
    
    left_bound = x + w*side_ratio
    right_bound = x + w*(1-side_ratio)
    
    ax.fill([x, x+w, x+w, x], [occl_y_start, occl_y_start, occl_y_end, occl_y_end],
            alpha=0.2, color=SURFACE_COLORS["Occlusal"])
    ax.fill([x, left_bound, left_bound, x], [y, y, y+h, y+h],
            alpha=0.2, color=SURFACE_COLORS["Mesial"])
    ax.fill([right_bound, x+w, x+w, right_bound], [y, y, y+h, y+h],
            alpha=0.2, color=SURFACE_COLORS["Distal"])
    
    # --- Draw tooth shape (light fill + outline) ---
    ax.fill(tooth_pts[:,0], tooth_pts[:,1], color="gray", alpha=0.7)  # ฟันสีอ่อน
    
    
    # --- Draw caries pixels on top ---
    color = SURFACE_COLORS.get(classification, '#95A5A6')
    if len(caries_pts) > 0:
        ax.scatter(caries_pts[:,0], caries_pts[:,1], c=color, s=25, edgecolors='black', alpha=0.2)
    
    # --- Draw centroid of tooth (star) ---
    c_cx, c_cy = compute_centroid(tooth_pts)
    ax.plot(c_cx, c_cy, '*', markersize=20, markeredgecolor='black', markerfacecolor='red')
    
    # Bounding box (optional)
    rect = Rectangle((x, y), w, h, fill=False, edgecolor='black', linestyle='--', linewidth=1.2)
    ax.add_patch(rect)
    
    ax.set_title(f"Tooth {tooth_id} - {classification}", fontsize=12)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    
    return fig

# =============================================================================
# Processing per case
# =============================================================================
def process_case(case_id):
    print(f"\n=== CASE {case_id} ===")
    seg_data = load_seg(case_id)
    caries_data = load_caries(case_id)
    if seg_data is None or caries_data is None: return

    seg_map = build_seg_map(seg_data)
    caries_list = get_caries_list(caries_data)

    results = {"case_number": case_id, "teeth_data": []}

    for t in caries_list:
        tid = str(t["tooth_id"])
        caries_pts = t.get("caries_coordinates", [])
        tooth_pts = seg_map.get(tid, [])
        if len(caries_pts) == 0 or len(tooth_pts) < 10: continue

        surface, angle = classify_surface_full(tid, tooth_pts, caries_pts)
        caries_pixels, percentage = compute_caries_stats(caries_pts, tooth_pts)

        results["teeth_data"].append({
            "tooth_id": tid,
            "has_caries": True,
            "confidence": t.get("confidence", 0),
            "caries_position_detail": surface,
            "rotation_angle": round(angle,2),
            "tooth_coordinates": tooth_pts,
            "caries_coordinates": caries_pts,
            "caries_pixels": caries_pixels,
            "caries_percentage": round(percentage,4)
        })

    # Save JSON
    case_dir = os.path.join(OUT_DIR, f"case_{case_id}")
    os.makedirs(case_dir, exist_ok=True)
    json_path = os.path.join(case_dir, f"case_{case_id}.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)
    print(f"[DONE] saved → {case_dir}")

def process_case_visual(case_id):
    case_dir = Path(OUT_DIR) / f"case_{case_id}"
    case_dir.mkdir(parents=True, exist_ok=True)

    json_path = Path(OUT_DIR) / f"case_{case_id}" / f"case_{case_id}.json"
    if not json_path.exists():
        print(f"[SKIP] Case {case_id} JSON not found")
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    # Save JSON copy
    json_save_path = case_dir / f"case_{case_id}.json"
    with open(json_save_path, "w") as f:
        json.dump(data, f, indent=4)

    # Create per-tooth visuals
    for t in data["teeth_data"]:
        tooth_id = t["tooth_id"]
        tooth_pts = t.get("tooth_coordinates", [])
        caries_pts = t.get("caries_coordinates", [])
        classification = t.get("caries_position_detail", "Other")

        tooth_dir = case_dir / f"tooth_{tooth_id}"
        tooth_dir.mkdir(parents=True, exist_ok=True)
        save_path = tooth_dir / f"tooth_{tooth_id}_visual.png"
        visualize_tooth_with_zones(tooth_pts, caries_pts, tooth_id, classification, save_path=save_path)

    print(f"[DONE] JSON + visuals saved for Case {case_id}")

# =============================================================================
# RUN EXAMPLES
# =============================================================================
# Process and visualize case 1
# process_case(1)
# process_case_visual(1)

# --- Run all cases (uncomment to process all) ---
for cid in range(1, 501):
    process_case(cid)
    process_case_visual(cid)


=== CASE 1 ===
[ERROR] SEG missing: segmentation+recognition-dataset/case 1/case_1_results.json
[ERROR] CARIES missing: caries_mapping_output/case 1/case_1_caries_mapping.json
[SKIP] Case 1 JSON not found

=== CASE 2 ===
[ERROR] SEG missing: segmentation+recognition-dataset/case 2/case_2_results.json
[ERROR] CARIES missing: caries_mapping_output/case 2/case_2_caries_mapping.json
[SKIP] Case 2 JSON not found

=== CASE 3 ===
[ERROR] SEG missing: segmentation+recognition-dataset/case 3/case_3_results.json
[ERROR] CARIES missing: caries_mapping_output/case 3/case_3_caries_mapping.json
[SKIP] Case 3 JSON not found

=== CASE 4 ===
[ERROR] SEG missing: segmentation+recognition-dataset/case 4/case_4_results.json
[ERROR] CARIES missing: caries_mapping_output/case 4/case_4_caries_mapping.json
[SKIP] Case 4 JSON not found

=== CASE 5 ===
[ERROR] SEG missing: segmentation+recognition-dataset/case 5/case_5_results.json
[ERROR] CARIES missing: caries_mapping_output/case 5/case_5_caries_mapping.json

## XML Parse Coverage
เช็คว่า xml ที่แปลงมาเป็น GT ใช้ได้จริง

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

# =========================================================
# Namespace
# =========================================================
AIM_NS = "gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM"
ISO_NS = "uri:iso.org:21090"
NS = {"aim": AIM_NS, "iso": ISO_NS}

# =========================================================
# Surface map
# =========================================================
SNODENT_SURFACE_MAP = {
    "144414D": "Occlusal",
    "146014D": "Distal",
    "145374D": "Mesial",
    "144474D": "Occlusal",
    "146074D": "Distal",
    "145434D": "Mesial",
}

DISPLAY_NAME_TO_SURFACE = {
    "Occlusal surface": "Occlusal",
    "Occlusal Surface": "Occlusal",
    "Distal Surface": "Distal",
    "Distal surface": "Distal",
    "Mesial Surface": "Mesial",
    "Mesial surface": "Mesial",
}

# =========================================================
# Tooth fallback map
# =========================================================
SNODENT_TO_FDI = {
    "161006D": "11",
    "160842D": "12",
    "160288D": "13",
    "161286D": "14",
    "160450D": "15",
    "161204D": "17",
    "160618D": "18",
    "160194D": "21",
    "160132D": "22",
    "160506D": "23",
    "161340D": "24",
    "160682D": "25",
    "161074D": "26",
    "160386D": "27",
    "160922D": "28",
    "161136D": "31",
    "160556D": "32",
    "160068D": "33",
    "160326D": "34",
    "161248D": "35",
    "160730D": "36",
    "161166D": "37",
    "160580D": "38",
    "160964D": "41",
    "160350D": "42",
    "160894D": "43",
    "160230D": "44",
    "161412D": "45",
    "161102D": "47",
    "160488D": "48",
}

# =========================================================
# Helpers
# =========================================================
def _get_display_name(element):
    dn = element.find("iso:displayName", NS)
    if dn is not None:
        return dn.get("value", "")
    return ""

def snodent_display_to_fdi(display_name):

    dn = display_name.lower()

    if "upper" in dn and "right" in dn:
        quadrant = 1
    elif "upper" in dn and "left" in dn:
        quadrant = 2
    elif "lower" in dn and "left" in dn:
        quadrant = 3
    elif "lower" in dn and "right" in dn:
        quadrant = 4
    else:
        return ""

    if "central incisor" in dn:
        pos = 1
    elif "lateral incisor" in dn:
        pos = 2
    elif "canine" in dn:
        pos = 3
    elif "first premolar" in dn:
        pos = 4
    elif "second premolar" in dn:
        pos = 5
    elif "first molar" in dn:
        pos = 6
    elif "second molar" in dn:
        pos = 7
    elif "third molar" in dn:
        pos = 8
    else:
        return ""

    return f"{quadrant}{pos}"

# =========================================================
# XML parser
# =========================================================
def parse_aim_xml(xml_path):

    try:
        tree = ET.parse(xml_path)
    except:
        return None

    root = tree.getroot()

    annotations = root.find("aim:imageAnnotations", NS)
    if annotations is None:
        return None

    ann = annotations.find("aim:ImageAnnotation", NS)
    if ann is None:
        return None

    tooth = ""
    surface = ""

    phys_coll = ann.find("aim:imagingPhysicalEntityCollection", NS)

    if phys_coll is not None:

        entity = phys_coll.find("aim:ImagingPhysicalEntity", NS)

        if entity is not None:

            char_coll = entity.find(
                "aim:imagingPhysicalEntityCharacteristicCollection", NS
            )

            if char_coll is not None:

                chars = char_coll.findall(
                    "aim:ImagingPhysicalEntityCharacteristic", NS
                )

                for ch in chars:

                    q_idx_el = ch.find("aim:questionIndex", NS)
                    q_idx = q_idx_el.get("value", "") if q_idx_el is not None else ""

                    tc = ch.find("aim:typeCode", NS)
                    if tc is None:
                        continue

                    code = tc.get("code", "")
                    display = _get_display_name(tc)

                    if q_idx == "0":

                        tooth = snodent_display_to_fdi(display)

                        if not tooth:
                            tooth = SNODENT_TO_FDI.get(code, "")

                    elif q_idx == "1":

                        surface = SNODENT_SURFACE_MAP.get(code, "")

                        if not surface:
                            surface = DISPLAY_NAME_TO_SURFACE.get(display, "")

    return {
        "tooth": tooth,
        "surface": surface
    }

# =========================================================
# COVERAGE CHECK
# =========================================================
base_path = Path("raw_data/500 cases with annotation(xml)")

ok = 0
missing_tooth = 0
missing_surface = 0
failed = 0

for case_folder in sorted(base_path.glob("case *")):

    for xml_file in case_folder.glob("*.xml"):

        parsed = parse_aim_xml(str(xml_file))

        if parsed is None:
            failed += 1
            continue

        if not parsed["tooth"]:
            missing_tooth += 1

        elif not parsed["surface"]:
            missing_surface += 1

        else:
            ok += 1

print("========== PARSE COVERAGE ==========")
print("OK lesions      :", ok)
print("Missing tooth   :", missing_tooth)
print("Missing surface :", missing_surface)
print("Failed XML      :", failed)

========== PARSE COVERAGE ==========
OK lesions      : 1979
Missing tooth   : 0
Missing surface : 0
Failed XML      : 0


## 5.Evaluation

parse XML ground truth + parse prediction refactor mapping + parser 
ทำ evaluation:
Accuracy
Precision
Recall
F1-score
Confusion Matrix

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

VALID_SURFACES = ["Occlusal", "Mesial", "Distal"]

# =========================================================
# LOAD GROUND TRUTH FROM XML
# =========================================================
def parse_case_ground_truth(case_folder):

    caries = []

    for xml_file in sorted(Path(case_folder).glob("*.xml")):

        parsed = parse_aim_xml(str(xml_file))

        if parsed is None:
            continue

        if parsed["surface"] in VALID_SURFACES:
            caries.append(parsed)

    return caries


# =========================================================
# LOAD PREDICTION FROM PCA OUTPUT
# =========================================================
def load_prediction(case_num):

    pred_path = Path("PCA_Output") / f"case_{case_num}" / f"case_{case_num}.json"

    if not pred_path.exists():
        return []

    with open(pred_path, "r") as f:
        data = json.load(f)

    preds = []

    for t in data.get("teeth_data", []):

        tooth = str(t.get("tooth_id", ""))
        surface = t.get("caries_position_detail", "")

        if surface in VALID_SURFACES:

            preds.append({
                "tooth": tooth,
                "surface": surface
            })

    return preds


# =========================================================
# MATCH GT VS PREDICTION
# =========================================================
def match_case(gt, pred):

    used_pred = set()

    TP = 0
    y_true = []
    y_pred = []

    for g in gt:

        found_tooth = False

        for i, p in enumerate(pred):

            if i in used_pred:
                continue

            if g["tooth"] == p["tooth"]:

                used_pred.add(i)
                found_tooth = True

                y_true.append(g["surface"])
                y_pred.append(p["surface"])

                if g["surface"] == p["surface"]:
                    TP += 1

                break

    FP = len(pred) - len(used_pred)
    FN = len(gt) - len(used_pred)

    return TP, FP, FN, y_true, y_pred


# =========================================================
# MAIN LOOP
# =========================================================
TP_total = 0
FP_total = 0
FN_total = 0

GT_total = 0
Pred_total = 0

all_y_true = []
all_y_pred = []

base_gt = Path("raw_data/500 cases with annotation(xml)")

for case_num in range(1, 501):

    gt_folder = base_gt / f"case {case_num}"

    gt = parse_case_ground_truth(gt_folder)
    pred = load_prediction(case_num)

    if len(gt) == 0 and len(pred) == 0:
        continue

    TP, FP, FN, yt, yp = match_case(gt, pred)

    TP_total += TP
    FP_total += FP
    FN_total += FN

    GT_total += len(gt)
    Pred_total += len(pred)

    all_y_true.extend(yt)
    all_y_pred.extend(yp)


# =========================================================
# FINAL METRICS
# =========================================================
precision = TP_total / (TP_total + FP_total) if (TP_total + FP_total) else 0
recall = TP_total / (TP_total + FN_total) if (TP_total + FN_total) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0


# =========================================================
# PRINT FINAL EVALUATION
# =========================================================
print("\n========== FINAL EVALUATION ==========")
print("GT Total Caries   :", GT_total)
print("Pred Total Caries :", Pred_total)
print("Matched Caries    :", TP_total)
print("False Positive    :", FP_total)
print("False Negative    :", FN_total)

print("\nPrecision :", round(precision,4))
print("Recall    :", round(recall,4))
print("F1-score  :", round(f1,4))


# =========================================================
# CONFUSION MATRIX
# =========================================================
if len(all_y_true) > 0:

    cm = confusion_matrix(
        all_y_true,
        all_y_pred,
        labels=VALID_SURFACES
    )

    cm_df = pd.DataFrame(
        cm,
        index=VALID_SURFACES,
        columns=VALID_SURFACES
    )

    print("\n========== CONFUSION MATRIX ==========")
    print(cm_df)

    print("\n========== CLASSIFICATION REPORT ==========")
    print(
        classification_report(
            all_y_true,
            all_y_pred,
            labels=VALID_SURFACES
        )
    )


========== FINAL EVALUATION ==========
GT Total Caries   : 1979
Pred Total Caries : 1172
Matched Caries    : 224
False Positive    : 278
False Negative    : 1085

Precision : 0.4462
Recall    : 0.1711
F1-score  : 0.2474

========== CONFUSION MATRIX ==========
          Occlusal  Mesial  Distal
Occlusal        50      16      12
Mesial         276     103     107
Distal         174      85      71

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

    Occlusal       0.10      0.64      0.17        78
      Mesial       0.50      0.21      0.30       486
      Distal       0.37      0.22      0.27       330

    accuracy                           0.25       894
   macro avg       0.33      0.36      0.25       894
weighted avg       0.42      0.25      0.28       894

